# Seed suite2p traces from cellpose masks

given cellpose masks that were run on 2x or 4x upsampled images, downsample the masks
back to native resolution and use suite2p to extract fluorescence traces, neuropil, and
deconvolved spikes.

In [18]:
# !uv pip install jupyterlab-vim

In [5]:
from pathlib import Path
import numpy as np
import tifffile
from suite2p.extraction.extract import extraction_wrapper
from suite2p.extraction.dcnv import preprocess, oasis
from suite2p.io import BinaryFile
from suite2p import default_ops
from scipy.ndimage import zoom as scipy_zoom
from skimage.transform import resize as ski_resize

## paths and parameters

In [ ]:
masks_root = Path(r"E:\datasets\lbm\cjennings\LBMdata_fig2_eval_v1") # plane01, plane02 ...
data_root = Path(r"E:\datasets\lbm\cjennings\data") # zplane01_tp...

# which cellpose condition to use
condition = "cpsam_upsamp1655_d4"  # upsampled 4x

# planes to process (mask planes 01-30 map to data zplanes 01-05)
mask_planes = sorted(masks_root.glob("plane*"))
data_planes = sorted(data_root.glob("zplane*"))

print(f"condition: {condition}")
print(f"mask planes: {len(mask_planes)}, data planes: {len(data_planes)}")
print(f"mask planes: {[p.name for p in mask_planes[:5]]}...")
print(f"data planes: {[p.name for p in data_planes]}")

condition: cpsam_upsamp1655_d4
mask planes: 30, data planes: 5
mask planes: ['plane01', 'plane02', 'plane03', 'plane04', 'plane05']...
data planes: ['zplane01_tp00001-08440', 'zplane02_tp00001-08440', 'zplane03_tp00001-08440', 'zplane04_tp00001-08440', 'zplane05_tp00001-08440']


## helper: downsample masks to match binary shape

label masks are resized to the binary data's (Ly, Lx) using nearest-neighbor
interpolation to preserve integer labels.

In [7]:
def downsample_masks(masks, target_shape):
    """resize a label mask to `target_shape` using nearest-neighbor interpolation."""
    ds = ski_resize(
        masks,
        target_shape,
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    ).astype(masks.dtype)
    return ds


def masks_to_stat(masks):
    """convert dense label mask to suite2p stat list."""
    labels = np.unique(masks)
    labels = labels[labels != 0]
    stat = []
    for label in labels:
        ypix, xpix = np.nonzero(masks == label)
        stat.append({
            "ypix": ypix.astype(np.int32),
            "xpix": xpix.astype(np.int32),
            "lam": np.ones(len(ypix), dtype=np.float32) / len(ypix),
            "med": np.array([np.median(ypix), np.median(xpix)]),
            "npix": len(ypix),
            "radius": np.sqrt(len(ypix) / np.pi),
            "overlap": np.zeros(len(ypix), dtype=bool),
        })
    return stat

## test on a single plane

load a mask, downsample it, convert to stat, and verify shapes match the binary data.

In [8]:
# pick first mask plane and first data plane
mask_dir = mask_planes[0]
data_dir = data_planes[0]

# find mask tif for selected condition
mask_files = list(mask_dir.glob(f"*{condition}*masks.tif"))
assert len(mask_files) == 1, f"expected 1 mask file, got {mask_files}"
mask_file = mask_files[0]

# load ops to get native binary shape
ops = np.load(data_dir / "ops.npy", allow_pickle=True).item()
Ly, Lx = ops["Ly"], ops["Lx"]

# load upsampled mask and resize to binary shape
masks_up = tifffile.imread(mask_file)
masks_native = downsample_masks(masks_up, (Ly, Lx))

print(f"mask file: {mask_file.name}")
print(f"upsampled mask shape: {masks_up.shape}")
print(f"downsampled mask shape: {masks_native.shape}")
print(f"binary data shape: ({Ly}, {Lx})")
print(f"scale factor: {masks_up.shape[0]/Ly:.2f}x, {masks_up.shape[1]/Lx:.2f}x")
print(f"n_rois upsampled: {len(np.unique(masks_up)) - 1}")
print(f"n_rois downsampled: {len(np.unique(masks_native)) - 1}")

mask file: 91c592e52265__cpsam_upsamp1655_d4__masks.tif
upsampled mask shape: (2525, 2190)
downsampled mask shape: (1002, 725)
binary data shape: (1002, 725)
scale factor: 2.52x, 3.02x
n_rois upsampled: 8221
n_rois downsampled: 8221


In [10]:
masks_up

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2525, 2190), dtype=uint16)

## overlay masks on mean image

sanity check that the downsampled masks align with the binary data.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# mean image from ops
mean_img = ops.get("max_proj", ops.get("meanImg_chan2", None))
if mean_img is None:
    with BinaryFile(Ly=Ly, Lx=Lx, filename=str(data_dir / "data.bin")) as f:
        mean_img = f[0:100].mean(axis=0)
mean_img = np.asarray(mean_img, dtype=np.float32)

# random colormap for mask overlay
rng = np.random.default_rng(42)
colors = np.zeros((masks_native.max() + 1, 4))
colors[1:, :3] = rng.random((masks_native.max(), 3))
colors[1:, 3] = 0.45
colors[0] = [0, 0, 0, 0]
cmap = ListedColormap(colors)

# pick 3 zoom regions around ROI-dense areas
cy, cx = Ly // 2, Lx // 2
zoom_size = min(Ly, Lx) // 5
zoom_regions = [
    ("top-left", slice(20, 20 + zoom_size), slice(20, 20 + zoom_size)),
    ("center", slice(cy - zoom_size // 2, cy + zoom_size // 2),
               slice(cx - zoom_size // 2, cx + zoom_size // 2)),
    ("bottom-right", slice(Ly - 20 - zoom_size, Ly - 20),
                     slice(Lx - 20 - zoom_size, Lx - 20)),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 14), dpi=150)

# top row: full FOV
axes[0, 0].imshow(mean_img, cmap="gray")
axes[0, 0].set_title("mean image")

axes[0, 1].imshow(masks_native, cmap=cmap, interpolation="nearest")
axes[0, 1].set_title(f"masks ({len(np.unique(masks_native)) - 1} rois)")

axes[0, 2].imshow(mean_img, cmap="gray")
axes[0, 2].imshow(masks_native, cmap=cmap, interpolation="nearest")
axes[0, 2].set_title("overlay")

# draw zoom region boxes on the overlay
for name, ysl, xsl in zoom_regions:
    rect = plt.Rectangle(
        (xsl.start, ysl.start), xsl.stop - xsl.start, ysl.stop - ysl.start,
        linewidth=1.5, edgecolor="yellow", facecolor="none",
    )
    axes[0, 2].add_patch(rect)

# bottom row: zoom-ins
for i, (name, ysl, xsl) in enumerate(zoom_regions):
    axes[1, i].imshow(mean_img[ysl, xsl], cmap="gray")
    axes[1, i].imshow(masks_native[ysl, xsl], cmap=cmap, interpolation="nearest")
    axes[1, i].set_title(f"zoom: {name}")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

NameError: name 'ops' is not defined

## extract traces for single plane

convert downsampled masks to stat, then use suite2p's `extraction_wrapper` to compute
F, Fneu from the registered binary. finally run OASIS deconvolution for spks.

In [12]:
stat = masks_to_stat(masks_native)
print(f"n_rois in stat: {len(stat)}")

# use registered binary for extraction
reg_file = str(data_dir / "data.bin")
with BinaryFile(Ly=Ly, Lx=Lx, filename=reg_file) as f_reg:
    n_frames = f_reg.shape[0]
    print(f"binary: {n_frames} frames, ({Ly}, {Lx})")

    s2p_ops = default_ops()
    s2p_ops.update({
        "Ly": Ly,
        "Lx": Lx,
        "batch_size": 500,
        "allow_overlap": True,
        "neucoeff": 0.7,
        "inner_neuropil_radius": 2,
        "min_neuropil_pixels": 350,
        "neuropil_extract": True,
    })

    stat, F, Fneu, _, _ = extraction_wrapper(stat, f_reg, ops=s2p_ops)

print(f"F shape: {F.shape}")
print(f"Fneu shape: {Fneu.shape}")

n_rois in stat: 8221
binary: 8440 frames, (1002, 725)
Masks created, 9.52 sec.


c:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\suite2p\extraction\extract.py:125: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  Fi[n] = np.dot(data[:, cell_ipix[n]], cell_lam[n])


Extracted fluorescence from 8221 ROIs in 8440 frames, 48.60 sec.
F shape: (8221, 8440)
Fneu shape: (8221, 8440)


## Deconvolution (spks.npy)

In [13]:
fs = ops.get("fs")  # sampling rate from original ops
tau = s2p_ops.get("tau")

# neuropil subtraction
Fc = F - s2p_ops["neucoeff"] * Fneu

# baseline correction + deconvolution
Fc_pre = preprocess(
    Fc,
    baseline=s2p_ops.get("baseline", "maximin"),
    win_baseline=s2p_ops.get("win_baseline", 60.0),
    sig_baseline=s2p_ops.get("sig_baseline", 10.0),
    fs=fs,
)
spks = oasis(Fc_pre, batch_size=s2p_ops["batch_size"], tau=tau, fs=fs)

print(f"spks shape: {spks.shape}")
print(f"fs: {fs} Hz, tau: {tau} s")

spks shape: (8221, 8440)
fs: 10.0 Hz, tau: 1.0 s


## save results for single plane

In [14]:
out_dir = data_dir / f"seeded_{condition}"
out_dir.mkdir(exist_ok=True)

np.save(out_dir / "stat.npy", stat, allow_pickle=True)
np.save(out_dir / "F.npy", F)
np.save(out_dir / "Fneu.npy", Fneu)
np.save(out_dir / "spks.npy", spks)
np.save(out_dir / "iscell.npy", np.column_stack([
    np.ones(len(stat)),
    np.zeros(len(stat)),
]))

print(f"saved to {out_dir}")
print(f"  stat: {len(stat)} rois")
print(f"  F: {F.shape}")
print(f"  Fneu: {Fneu.shape}")
print(f"  spks: {spks.shape}")

saved to E:\datasets\lbm\cjennings\data\zplane01_tp00001-08440\seeded_cpsam_upsamp1655_d4
  stat: 8221 rois
  F: (8221, 8440)
  Fneu: (8221, 8440)
  spks: (8221, 8440)


## batch: process all matching planes

loops over data planes, finds the corresponding mask plane, downsamples, extracts, and saves.
mask planes 01-30 correspond to data zplanes 01-05 (adjust mapping if needed).

In [15]:
def process_plane(mask_dir, data_dir, condition):
    """process a single plane: resize masks to binary shape -> extract traces -> deconvolve -> save."""
    # find mask file
    mask_files = list(mask_dir.glob(f"*{condition}*masks.tif"))
    if len(mask_files) != 1:
        print(f"  skipping {mask_dir.name}: found {len(mask_files)} mask files")
        return None

    # load ops to get native shape
    ops = np.load(data_dir / "ops.npy", allow_pickle=True).item()
    Ly, Lx = ops["Ly"], ops["Lx"]

    # load and resize masks to binary shape
    masks_up = tifffile.imread(mask_files[0])
    masks_native = downsample_masks(masks_up, (Ly, Lx))

    # convert to stat
    stat = masks_to_stat(masks_native)
    if len(stat) == 0:
        print(f"  skipping {mask_dir.name}: no rois found")
        return None

    # extract traces
    reg_file = str(data_dir / "data.bin")
    s2p_ops = default_ops()
    s2p_ops.update({
        "Ly": Ly, "Lx": Lx, "batch_size": 500,
        "allow_overlap": True, "neucoeff": 0.7,
        "inner_neuropil_radius": 2, "min_neuropil_pixels": 350,
        "neuropil_extract": True,
    })

    with BinaryFile(Ly=Ly, Lx=Lx, filename=reg_file) as f_reg:
        stat, F, Fneu, _, _ = extraction_wrapper(stat, f_reg, ops=s2p_ops)

    # deconvolution
    fs = ops.get("fs", 30.0)
    tau = s2p_ops.get("tau", 1.0)
    Fc = F - s2p_ops["neucoeff"] * Fneu
    Fc_pre = preprocess(
        Fc, baseline="maximin", win_baseline=60.0, sig_baseline=10.0, fs=fs,
    )
    spks = oasis(Fc_pre, batch_size=500, tau=tau, fs=fs)

    # save
    out_dir = data_dir / f"seeded_{condition}"
    out_dir.mkdir(exist_ok=True)
    np.save(out_dir / "stat.npy", stat, allow_pickle=True)
    np.save(out_dir / "F.npy", F)
    np.save(out_dir / "Fneu.npy", Fneu)
    np.save(out_dir / "spks.npy", spks)
    np.save(out_dir / "iscell.npy", np.column_stack([
        np.ones(len(stat)), np.zeros(len(stat)),
    ]))

    return {"plane": mask_dir.name, "n_rois": len(stat), "n_frames": F.shape[1]}

## run batch

adjust the plane mapping below. currently assumes mask plane01 -> data zplane01, etc.
only processes planes where both mask and data directories exist.

In [16]:
# build plane mapping: mask plane## -> data zplane##
# only the first 5 data zplanes exist, so we pair mask plane01-05 with zplane01-05
plane_pairs = []
for dp in data_planes:
    plane_num = dp.name.replace("zplane", "").split("_")[0]  # e.g. "01"
    mp = masks_root / f"plane{plane_num}"
    if mp.exists():
        plane_pairs.append((mp, dp))

print(f"plane pairs to process: {len(plane_pairs)}")
for mp, dp in plane_pairs:
    print(f"  {mp.name} -> {dp.name}")

plane pairs to process: 5
  plane01 -> zplane01_tp00001-08440
  plane02 -> zplane02_tp00001-08440
  plane03 -> zplane03_tp00001-08440
  plane04 -> zplane04_tp00001-08440
  plane05 -> zplane05_tp00001-08440


In [17]:
results = []
for mp, dp in plane_pairs:
    print(f"processing {mp.name} -> {dp.name} ...")
    result = process_plane(mp, dp, condition)
    if result:
        results.append(result)
        print(f"  done: {result['n_rois']} rois, {result['n_frames']} frames")

print(f"\nfinished {len(results)}/{len(plane_pairs)} planes")

processing plane01 -> zplane01_tp00001-08440 ...
Masks created, 9.39 sec.
Extracted fluorescence from 8221 ROIs in 8440 frames, 37.79 sec.
  done: 8221 rois, 8440 frames
processing plane02 -> zplane02_tp00001-08440 ...
Masks created, 13.10 sec.
Extracted fluorescence from 11541 ROIs in 8440 frames, 49.65 sec.
  done: 11541 rois, 8440 frames
processing plane03 -> zplane03_tp00001-08440 ...
Masks created, 12.33 sec.
Extracted fluorescence from 10886 ROIs in 8440 frames, 47.06 sec.
  done: 10886 rois, 8440 frames
processing plane04 -> zplane04_tp00001-08440 ...
Masks created, 10.28 sec.
Extracted fluorescence from 8989 ROIs in 8440 frames, 45.39 sec.
  done: 8989 rois, 8440 frames
processing plane05 -> zplane05_tp00001-08440 ...
Masks created, 11.25 sec.
Extracted fluorescence from 9722 ROIs in 8440 frames, 47.72 sec.
  done: 9722 rois, 8440 frames

finished 5/5 planes
